# 🔁 Cross-Validation & Resampling
**ISLP Chapter 5 · Pattern Recognition for the Rest of Us**

> How do you honestly estimate how well your model will perform on new data — without collecting new data? Cross-validation is the answer, and it's one of the most practically important techniques in this entire course.

---
### What you'll learn
- Why a simple train/test split can be misleading
- Leave-one-out cross-validation (LOOCV) — the extreme case
- K-fold cross-validation — the standard in practice
- The bootstrap — estimating uncertainty in any statistic
- How to use CV for model selection

### Time
~55 minutes

## 🎯 Part 1 — The Problem with a Single Train/Test Split

When we split data randomly into training and test sets, the test error estimate depends on *which observations happened to land in the test set*. With small datasets this variance can be huge — you might get lucky or unlucky.

**Example:** With 100 observations and a 70/30 split, there are C(100,30) ≈ 3×10²⁵ possible test sets. Each gives a different error estimate.

Cross-validation solves this by systematically using *all* the data for both training and validation, averaging the results.

In [ ]:
# Use Adult Income dataset for CV demonstration
# Cross-validation works identically for classification — let's use a real business problem
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Reconstruct the Adult income features (numeric only for clean CV demo)
np.random.seed(42)
n = 4000
age = np.random.randint(18, 75, n)
edu = np.random.randint(1, 16, n)
hrs = np.random.randint(20, 65, n)
cap = np.random.choice([0]*9 + [5000], n)

log_odds = -3 + 0.04*age + 0.15*edu + 0.02*hrs
y_adult = (1/(1+np.exp(-log_odds)) > np.random.uniform(0,1,n)).astype(int)
X_adult = StandardScaler().fit_transform(
    np.column_stack([age, edu, hrs, cap]))

print(f"Adult income data: {X_adult.shape}  |  Income >50K rate: {y_adult.mean():.1%}")
print("Using KNN on Adult Income to demonstrate CV")


In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Auto

In [ ]:
import pandas as pd
auto = pd.read_csv(f'{DATA_DIR}/Auto.csv').dropna()
print(f"Auto: {auto.shape}  |  Columns: {list(auto.columns)}")
auto.head()

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

## 🔂 Part 2 — K-Fold Cross-Validation

**K-fold CV** divides the data into K equal-sized groups (folds). For each fold k:
1. Train on the other K-1 folds
2. Validate on fold k
3. Record the error

After K iterations, average the K error estimates.

```
CV(K) = (1/K) × Σ MSEₖ
```

**Typical values:** K=5 or K=10. K=10 is the most common in practice.

**Why not K=2?** High bias — each training set is only half the data.  
**Why not K=n (LOOCV)?** Very slow, high variance in some settings.

In [ ]:
# Visualize how K-fold CV works
fig, ax = plt.subplots(figsize=(11, 4))
n_obs, K = 30, 5
fold_size = n_obs // K

colors = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1','#0e7490']

for k in range(K):
    for i in range(n_obs):
        fold = i // fold_size
        if fold == k:
            color = colors[k]
            label = 'Validation' if i == k * fold_size else ''
        else:
            color = '#d0d8e8'
            label = ''
        ax.barh(k, 1, left=i, color=color, edgecolor='white', height=0.7)

ax.set_yticks(range(K))
ax.set_yticklabels([f'Fold {k+1}' for k in range(K)])
ax.set_xlabel('Observation index')
ax.set_title('5-Fold Cross-Validation — colored = validation fold, gray = training')

# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=f'Fold {i+1} as validation') for i in range(K)]
legend_elements.append(Patch(facecolor='#d0d8e8', label='Training'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 🔂 Part 3 — Leave-One-Out CV (LOOCV)

LOOCV is K-fold CV where K = n. Each observation gets its own validation fold.

**Pros:** Nearly unbiased estimate of test error (trained on n-1 observations ≈ full data).  
**Cons:** Must fit the model n times — expensive for large n. Can have high variance.

**When to use LOOCV:** Small datasets (n < 100) where you can't afford to leave much data out.

In [ ]:
# LOOCV vs 5-fold vs 10-fold — compare estimates and runtime
import time

results = {}
for name, cv in [('LOOCV', LeaveOneOut()),
                 ('5-Fold CV', KFold(5, shuffle=True, random_state=42)),
                 ('10-Fold CV', KFold(10, shuffle=True, random_state=42))]:
    pipe = Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())])
    t0 = time.time()
    scores = cross_val_score(pipe, X_auto, y_auto, cv=cv,
                             scoring='neg_mean_squared_error')
    elapsed = time.time() - t0
    results[name] = {'mse': -scores.mean(), 'std': scores.std(), 'time': elapsed}

print("=" * 55)
print(f"{'Method':<15} {'CV MSE':>10} {'Std':>10} {'Time (s)':>10}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<15} {r['mse']:>10.3f} {r['std']:>10.3f} {r['time']:>10.3f}")
print("=" * 55)
print("\n📌 All three give similar MSE estimates.")
print("   LOOCV is slower and has higher std — 10-fold is the standard choice.")

## 🥾 Part 4 — The Bootstrap

The bootstrap estimates the variability of *any* statistic by resampling with replacement from your dataset.

**How it works:**
1. Sample n observations with replacement → bootstrap dataset B*
2. Compute your statistic on B* → get θ̂*
3. Repeat B times (usually B=1000)
4. The standard deviation of {θ̂₁*, θ̂₂*, ..., θ̂ᴮ*} estimates the SE of your statistic

**Key insight:** "with replacement" means some observations appear multiple times, others not at all (~37% are never sampled). Each bootstrap dataset is a plausible alternative sample from the population.

In [ ]:
# Bootstrap example: estimate standard error of the median
np.random.seed(42)
# Simulate a skewed distribution (e.g., income)
population_sample = np.concatenate([
    np.random.exponential(scale=50000, size=200),
    np.random.normal(200000, 30000, 20)   # a few high earners
])
population_sample = population_sample[population_sample > 0]
n = len(population_sample)

# Bootstrap
B = 2000
bootstrap_medians = []
for _ in range(B):
    boot_sample = np.random.choice(population_sample, size=n, replace=True)
    bootstrap_medians.append(np.median(boot_sample))

se_bootstrap = np.std(bootstrap_medians)
observed_median = np.median(population_sample)
ci_lower, ci_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: data distribution
axes[0].hist(population_sample, bins=40, color='#1e5fa8', alpha=0.7, edgecolor='white')
axes[0].axvline(observed_median, color='#e85d20', lw=2, label=f'Median = ${observed_median:,.0f}')
axes[0].set_title('Income Distribution (n=220)')
axes[0].set_xlabel('Income ($)')
axes[0].legend()

# Right: bootstrap distribution of median
axes[1].hist(bootstrap_medians, bins=50, color='#1a7a45', alpha=0.7, edgecolor='white')
axes[1].axvline(observed_median, color='#e85d20', lw=2, label=f'Observed median')
axes[1].axvline(ci_lower, color='#888', lw=1.5, ls='--')
axes[1].axvline(ci_upper, color='#888', lw=1.5, ls='--', label=f'95% CI: [${ci_lower:,.0f}, ${ci_upper:,.0f}]')
axes[1].set_title(f'Bootstrap Distribution of Median (B={B})')
axes[1].set_xlabel('Bootstrap Median ($)')
axes[1].legend(fontsize=9)

plt.suptitle('Bootstrap — Estimating Uncertainty in the Median', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nObserved median:    ${observed_median:,.0f}")
print(f"Bootstrap SE:       ${se_bootstrap:,.0f}")
print(f"95% CI:             [${ci_lower:,.0f}, ${ci_upper:,.0f}]")
print("\n📌 We never collected new data — the bootstrap created 2000 synthetic datasets from the one we have.")

In [ ]:
# Use Adult Income dataset for CV demonstration
# Cross-validation works identically for classification — let's use a real business problem
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Reconstruct the Adult income features (numeric only for clean CV demo)
np.random.seed(42)
n = 4000
age = np.random.randint(18, 75, n)
edu = np.random.randint(1, 16, n)
hrs = np.random.randint(20, 65, n)
cap = np.random.choice([0]*9 + [5000], n)

log_odds = -3 + 0.04*age + 0.15*edu + 0.02*hrs
y_adult = (1/(1+np.exp(-log_odds)) > np.random.uniform(0,1,n)).astype(int)
X_adult = StandardScaler().fit_transform(
    np.column_stack([age, edu, hrs, cap]))

print(f"Adult income data: {X_adult.shape}  |  Income >50K rate: {y_adult.mean():.1%}")
print("Using KNN on Adult Income to demonstrate CV")


## 💼 Exercise

**Task 1:** Load the Auto dataset. Use 10-fold CV to compare Linear, Ridge (alpha=1), and Lasso (alpha=1) regression predicting `mpg` from all numeric features. Which has the best CV MSE?

**Task 2:** On a dataset of your choice, compare the CV estimates from 5-fold, 10-fold, and LOOCV. How different are they? At what n does LOOCV become impractical?

**Task 3:** Use the bootstrap to estimate the standard error of the *mean* horsepower in the Auto dataset. How does your bootstrap SE compare to the analytical SE (std/√n)?

In [ ]:
answers = {
    "q1": "",  # What does K-fold CV do that a single train/test split does not?
    "q2": "",  # Why is LOOCV sometimes worse than 10-fold CV despite being less biased?
    "q3": "",  # In bootstrap resampling, approximately what fraction of observations are never sampled?
    "q4": "",  # You have 20 observations. Which CV method would you choose and why?
    "q5": "",  # How does CV help with model selection? Give a specific example.
}
missing = [k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ All answered! Save: File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Cross-Validation & Resampling"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
